# Team 13: Module 4 - Employer project for Lendable 
## Data validation and warehouse ETL process notebook

This notebook contains a proof-of-concept (POC) for a data import and validation process designed to 
be used to take data from the loan management system at Lendable and import it into a                
cloud data warehouse.  As the project team cannot be given access to these systems we have "mocked" 
the methods which will import the data to just pick up the data from the tabs of                   
the spreadsheet supplied by Lendable.                                                                
To avoid getting into implementation and deployment detail in a POC we have used a local Postgres    
database to represent the cloud data warehouse                                                       
The intention is that these four methods could be rewritten to take the data from Lendables systems
and that the data warehouse connection parameters could be modified to use a cloud data platform as opposed to the local Postgres database used for the POC.

We define methods below which are used to extract the source data, validate it and produce details of the issues 
and to manage the staging tables used to store the data during validation.

In [63]:
########################################################################################################
#Version history
# Jonathan Shields 29/08/2025                 Initial version
########################################################################################################

#Imports
import pandas as pd
import numpy as np
import re
from datetime import datetime
from collections import namedtuple
from sqlalchemy.engine import result #SQLAlchemy used as it is widely used and well documented
from sqlalchemy import insert
from sqlalchemy import update
from sqlalchemy import MetaData
from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy import Table
from sqlalchemy import quoted_name
from sqlalchemy.orm import sessionmaker
import configparser

#For the purposes of this project the Excel spreadsheet provided by Lendable will be the source data.  This can be replaced
#with connections to loan management systems when Lendable take the project forward

excelFilePath=r"C:\Users\jps16\OneDrive\Documents\Career Accelerator\Employer Project\Data\Lendable Anon Data Version 2.xlsx"
#Names f tabs in Excel file for the four sources
portfolioTabName="Data_Portfolio Anon"
couponTabName="Data_Coupon Anon"
borrowerTabName="Borrower_Identifiers Anon"
valuationTabName="Data_Valuation"

#Path to ISO country codes sheet
isoCountryExcelPath=r"C:\Users\jps16\OneDrive\Documents\Career Accelerator\Employer Project\Data\ISOCountryCodes.xlsx"

#Strings to represent the four sources - used to avoid hardcoding these everywhere
portfolioSourceName="portfolio"
portfolioIssueTable="portfolio_data_issues"
portfolioIssueDetailTable="portfolio_data_issue_details"
couponSourceName="coupon"
couponIssueTable="coupon_data_issues"
couponIssueDetailTable="coupon_data_issue_details"
borrowerSourceName="borrower"
borrowerIssueTable="borrower_data_issues"
borrowerIssueDetailTable="borrower_data_issue_details"
valuationSourceName="valuation"
valuationIssueTable="valuation_data_issues"
valuationIssueDetailTable="valuation_data_issue_details"

#Names of data quality/staging schemas
dataQualitySchema="data_quality"
stagingSchema="staging"

#Read config file to retrieve connection string parameters - in cloud data platform could replace with local variables
appConfig = configparser.ConfigParser()
appConfig.read("App.ini")

server=appConfig.get("WarehouseDB","host")
user=appConfig.get("WarehouseDB","user")
password=appConfig.get("WarehouseDB","password")
database=appConfig.get("WarehouseDB","database")

#Connection string for our database
#Will need to be changed to use the data warehouse selected rather than the Postgres database for this POC

connect_string=rf"postgresql+psycopg2://{user}:{password}@{server}/{database}"

#Specify maximum number of countries for each borrower record (currently 7)
borrowerMaxCountries=7

#Specify number of borrowers for each date row (ie columns) in the valuation data
valuationBorrowerCols=4

#Define functions to return 4 dataframes containing each dataset

def getPortfolioData() -> pd.DataFrame:
    
    #TO DO: replace with code to extract data from loan management systems
    #Get a dataframe containing the portfolio data
    df=pd.read_excel(excelFilePath,portfolioTabName)
    
    return df
    
def getCouponData() -> pd.DataFrame:
    #Get a dataframe containing the coupon data
    #TO DO: replace with code to extract data from loan management systems
    df=pd.read_excel(excelFilePath,couponTabName)

    return df
    
def getBorrowerData() -> pd.DataFrame:
    #Get a dataframe containing the borrower data
    #TO DO: replace with code to extract data from loan management systems
    df=pd.read_excel(excelFilePath,borrowerTabName)

    
    return df

def getValuationData() -> pd.DataFrame:
    #Get a dataframe containing the borrower data

    #TO DO: replace with code to extract data from loan management systems
    #Formatting below is just to handle the Excel sheet

    names=["Date","RocketLend","OmniFund","CargoSpeed","AlphaLend","HorizonCapital"]
    
    #Assign column names as the first rows are blank...
    df=pd.read_excel(excelFilePath,valuationTabName,header=None,names=names)

    #Remove unwanted non-dated rows
    df=df.loc[df["Date"].apply(validateReportDate)==""]
    return df

def getISOCountryData() -> pd.DataFrame:
    #Return a list of valid country names and ISO codes
    #We are using a spreadsheet for this in the POC
    
    df=pd.read_excel(isoCountryExcelPath,"ISO country codes")

    df.replace('', np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

#Function to clear down all of our tables (except process_summaries)
def clearDownData():

    sqlStatements=[]

    #Data quality tables
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{valuationIssueDetailTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{valuationIssueTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{portfolioIssueDetailTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{portfolioIssueTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{couponIssueDetailTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{couponIssueTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{borrowerIssueDetailTable}")
    sqlStatements.append(f"DELETE FROM {dataQualitySchema}.{borrowerIssueTable}")

    #Staging tables
    sqlStatements.append(f"DELETE FROM {stagingSchema}.portfolios")
    sqlStatements.append(f"DELETE FROM {stagingSchema}.coupons")
    sqlStatements.append(f"DELETE FROM {stagingSchema}.borrowers")
    sqlStatements.append(f"DELETE FROM {stagingSchema}.valuations")

    engine=create_engine(connect_string)
    
    with engine.begin() as connection:
        # a sessionmaker(), also in the same scope as the engine
        Session = sessionmaker(engine)

        # we can now construct a Session() and include begin()/commit()/rollback() automatically
        # automatically...see https://docs.sqlalchemy.org/en/14/orm/session_basics.html#framing-out-a-begin-commit-rollback-block
        #This is so that if one delete fails, they all rollback so we don't end up with a half-cleared scenario

        with Session.begin() as session:
            for statement in sqlStatements:
                session.execute(text(statement))

#Function to validate portfolio data
#df - the dataframe we are cleansing
#sourceTypeName which source are we cleansing...Portfolio,Coupon,Borrower,Valuation
#processSummary id so we can use it to link the issues
#country_df - iso country data (used for borrower validation only)

#Returns count of rows with issues 
def performValidation(df:pd.DataFrame,sourceTypeName:str,processSummaryId:int,country_df:pd.DataFrame=None)->int:

    #Applies validation rules and creates two dataframes
    #The first contains a row for each row in the source table with an issue
    #The second has a "foreign key" to the first and contains a row per field with an issue in that row

    #Rename the columns to not have spaces so we can use itertuples later and in the db.
    #\s+ is a regular expression meaning any number of spaces, which will be replaced with an underscore
    df.columns = df.columns.str.replace(r'\s+', '_', regex=True) 

    #Remove brackets and ?
    df.columns = df.columns.str.replace('(','')
    df.columns = df.columns.str.replace(')','')
    df.columns = df.columns.str.replace('?','')

    #Replace hyphen with underscore
    df.columns = df.columns.str.replace('-','_')
    
    #Make all columns lowercase named to avoid issues with Postgres
    df.columns = map(str.lower, df.columns)

    #Create 2 new DataFrames to store the data quality output
    
    #Issues df is an empty copy of the frame with the data
    issues_df=df.iloc[:0].copy()

    #Add an empty id column at the start which will be used for storing our PK id
    issues_df.insert(0, 'id',value=0)
    #Add the process summary field
    issues_df.insert(1, 'process_summary_id',value=0)

    #name of the column in the issue details which will be our database foreign key to the issue table
    issuedesc_foreignkey=f"{sourceTypeName.lower()}_data_issue_id" 
    
    #Create our issue description dataframe, which will contain one row for each field in the record that has a problem
    issuedesc_df=pd.DataFrame(columns=[issuedesc_foreignkey,"field_name","description"])

    issues=0

    #Call the relevant method to validate the data source depending on the type
    if sourceTypeName==portfolioSourceName:
        issues=performPortfolioValidation(df,issues_df,issuedesc_df,processSummaryId)
    elif sourceTypeName==couponSourceName:
        issues=performCouponValidation(df,issues_df,issuedesc_df,processSummaryId)
    elif sourceTypeName==borrowerSourceName:
        #Borrower validation requires the country data
        issues=performBorrowerValidation(df,issues_df,issuedesc_df,processSummaryId,country_df)
    elif sourceTypeName==valuationSourceName:
        issues=performValuationValidation(df,issues_df,issuedesc_df,processSummaryId)

    return issues

def performPortfolioValidation(portfolio_df:pd.DataFrame,issues_df:pd.DataFrame,issuedesc_df:pd.DataFrame,processSummaryId:int)->int:

    issueId=1

    #Clean NaN out of transfers column
    portfolio_df.loc[portfolio_df["transfers"].isna(),"transfers"]=""
    
    #Loop around the rows, using itertuples for performance
    for row in portfolio_df.itertuples():

        issueRowInserted=False
        
        if row.borrower.strip()=="":
            #Borrower must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "portfolio_data_issue_id","Borrower","MissingValue",processSummaryId)
        if row.facility_name.strip()=="":
            #Facility must be non-blank
           (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                               "portfolio_data_issue_id","Facility Name","MissingValue",processSummaryId)
        if row.deal_name.strip()=="":
            #Deal must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Deal Name","MissingValue",processSummaryId)
        if row.transfers.strip()!="" and row.transfers.strip().upper()!="Y":
            #Transfers can only be Y or blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Transfers?","ValueOutOfRange",processSummaryId)
        if row.fund.strip()=="":
            #Fund must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Fund","MissingValue",processSummaryId)
        if validateReportDate(row.date)!="":
            #Date must be dd/mm/yyyy format and must fall between 01/01/1970 and 01/01/2050
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Date",validateReportDate(row.date),processSummaryId)
        if row.segment.strip()=="":
            #Segment must be non-blank
            issueRowInserted=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Segment","MissingValue",processSummaryId)
        if np.isnan(row.balance_gross):
            #Balance must have a value
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Segment","MissingValue",processSummaryId)
        
        if not isinstance(row.balance_gross, (int, float)):
            #Balance must be numeric
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                       "portfolio_data_issue_id","Balance_gross","InvalidNumberFormat",processSummaryId)
        elif float(row.balance_gross)<0 or float(row.balance_gross)>100000000:
            #Balance must be > 0 and < 100 million
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                      "portfolio_data_issue_id","Balance_gross","ValueOutOfRange",processSummaryId)
        
        if not isinstance(row.principal_additions, (int, float)):
            #Principal additions must be numeric
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "portfolio_data_issue_id","Principal_Additions",
                                                                                "InvalidNumberFormat",processSummaryId)
        elif float(row.principal_additions)<0 or float(row.principal_additions)>100000000:
            #and must be > 0 and < 100 million
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "portfolio_data_issue_id","Principal_Additions",
                                                                                "ValueOutOfRange",processSummaryId)

        if issueRowInserted:
            issueId+=1

    #Now that we have processed all the rows, look for duplicates on borrower/facility/deal/date
    #Only using itertuples here as it is compatible with our lower level method insertIssueInto...
    
    for duplicate in portfolio_df[portfolio_df.duplicated(subset=['borrower','facility_name','deal_name','date'],keep='last')].itertuples():

        #Insert into issue tables as before, but detail table will have a single row with a blank fieldname
        (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,False,duplicate,issueId,
                                                                            "portfolio_data_issue_id","","Duplicate",processSummaryId)
        
        if issueRowInserted:
            issueId+=1
            
    #We now have our errors saved into our 2 dataframes - 
    #Save them into our SQL tables

    engine= create_engine(connect_string)
    
    if issues_df.shape[0]>0:

        #Save issues and detail/description tables to db
        issues_df.to_sql(portfolioIssueTable, engine, index=False,if_exists='append',schema='data_quality')
        issuedesc_df.to_sql(portfolioIssueDetailTable,engine, index=False,if_exists='append',schema='data_quality')

    #IssueId is also a count of how many issues we found (if we subtract 1)
    return issueId-1


We now define similar functions to validation coupon, valuation and borrower data in the same way to the portfolio data.

In [65]:
#Function to validate coupon data
def performCouponValidation(coupon_df:pd.DataFrame,issues_df:pd.DataFrame,issuedesc_df:pd.DataFrame,processSummaryId:int)->int:

    issueId=1

    #Loop around the rows
    for row in coupon_df.itertuples():

        issueRowInserted=False
        
        if row.borrower.strip()=="":
            #Borrower must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id","Borrower","MissingValue",processSummaryId)
            
        #For some odd reason, the dealname shows as a float even though in Excel it is blank and there are no numeric dealnames in the file....
        if isinstance(row.deal_name,(int,float)) or row.deal_name.strip()=="":
            #Deal must be non-blank
            issueRowInserted=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,"coupon_data_issue_id",
                                                       "Deal Name","MissingValue",processSummaryId)
        if validateReportDate(row.date_eom)!="":
            
            #Date must be dd/mm/yyyy format and must fall between 01/01/1970 and 01/01/2050
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id","Date",validateReportDate(row.date_eom)
                                                                                ,processSummaryId)
        if row.coupon_category.strip()=="":
            #Must be non-blank - do not go further than this as we do not know what grouping people may wish to use in future
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id", "Coupon Category",
                                                                                "MissingValue",processSummaryId)
        if validateReportDate(row.maturity)!="":
            #Maturity must be dd/mm/yyyy format and must fall between 01/01/1970 and 01/01/2050
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id","Maturity",
                                                                                validateReportDate(row.maturity),processSummaryId)    
            
        if not isinstance(row.balance_fv, (int, float)):
            #Balance must be numeric
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id","Balance_FV",
                                                                                "InvalidNumberFormat",processSummaryId)
        elif float(row.balance_fv)<0 or float(row.balance_fv)>100000000:
            #Balance must be > 0 and < 100 million
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "coupon_data_issue_id", "Balance_FV",
                                                                                "ValueOutOfRange",processSummaryId)

        if validatePercentage(row.coupon)!="":
            #Coupon percentage must be in a valid format
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,"Coupon",
                                                                                validatePercentage(row.coupon),processSummaryId)  

        if issueRowInserted:
            issueId+=1

    #Now that we have processed all the rows, look for duplicates on borrower/deal/date (keeping the last of the pair of dupes)

    for duplicate in coupon_df[coupon_df.duplicated(subset=['borrower','deal_name','date_eom'],keep='last')].itertuples():

        #Insert into issue tables as before, but detail table will have a single row with a blank fieldname
        (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,False,duplicate,issueId,"coupon_data_issue_id",
                                                                            "","Duplicate",processSummaryId)
        
        if issueRowInserted:
            issueId+=1

    #We now have our errors saved into our 2 dataframes - 
    #Save them into our SQL tables
    
    if issues_df.shape[0]>0:
        engine=create_engine(connect_string)
        
        #Save issues and detail/description tables to db
        issues_df.to_sql(couponIssueTable, engine, index=False,if_exists='append',schema='data_quality')
        issuedesc_df.to_sql(couponIssueDetailTable,engine, index=False,if_exists='append',schema='data_quality')

     #Return a count of how many issues we found (if we subtract 1)
    return issueId-1

In [66]:
#Function to validate valuation data

def performValuationValidation(valuation_df:pd.DataFrame,issues_df:pd.DataFrame,issuedesc_df:pd.DataFrame,processSummaryId:int)->int:

    issueId=1
    
    #Loop around the rows
    for row in valuation_df.itertuples():

        issueRowInserted=False
        
        if validateReportDate(row.date)!="":
            #Date must be dd/mm/yyyy format and must fall between 01/01/1970 and 01/01/2050
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "valuation_data_issue_id","Date",
                                                                                validateReportDate(row.date),processSummaryId)

        #All other columns having values must be percentages

        for i in range(2,valuationBorrowerCols):
            #All need to be valid percentages
            if validatePercentage(row[i])!="":
                (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                    "valuation_data_issue_id","",
                                                                                    validatePercentage(row[i]),processSummaryId)

        if issueRowInserted:
            issueId+=1

    #Now that we have processed all the rows, look for duplicates on date

    for duplicate in valuation_df[valuation_df.duplicated(subset=['date'],keep='last')].itertuples():

        #Insert into issue tables as before, but detail table will have a single row with a blank fieldname
        (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,False,duplicate,issueId,
                                                                            "valuation_data_issue_id","","Duplicate",processSummaryId)

        if issueRowInserted:
            issueId+=1

    #We now have our errors saved into our 2 dataframes - 
    #Save them into our SQL tables
    
    if issues_df.shape[0]>0:

        engine=create_engine(connect_string)
        
        #Save issues and detail/description tables to db
        issues_df.to_sql(valuationIssueTable, engine, index=False,if_exists='append',schema='data_quality')
        issuedesc_df.to_sql(valuationIssueDetailTable,engine, index=False,if_exists='append',schema='data_quality')

    #Return count of issue rows inserted
    return issueId-1;


In [67]:
#Function to validate borrower data
def performBorrowerValidation(borrower_df:pd.DataFrame,issues_df:pd.DataFrame,issuedesc_df:pd.DataFrame,processSummaryId:int,
                              country_df:pd.DataFrame)->int:

    issueId=1

    #Get a list of the valid countries

    valid_countries=[country.strip().lower() for country in country_df["English short name"].unique()]
    
    #Loop around the rows
    for row in borrower_df.itertuples():

        issueRowInserted=False
        
        if row.borrower.strip()=="":
            #Borrower must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                  "borrower_data_issue_id","Borrower","MissingValue",processSummaryId)

        if not isinstance(row.risk_rating, (int)):
            #Risk rating must be an integer
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                  "borrower_data_issue_id","Risk rating","InvalidNumberFormat",processSummaryId)
        elif int(row.risk_rating)<1 or int(row.risk_rating)>8:
            #Risk rating must be > 0 and < 9
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                      "borrower_data_issue_id","Risk rating","ValueOutOfRange",processSummaryId)

        if row.type_loan.strip().lower() not in ['secured','unsecured','','-']:
            #Allow secured,unsecured,blank and dash for type load
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "borrower_data_issue_id","Type loan",
                                                                                "ValueOutOfRange",processSummaryId)

        if row.segment.strip()=="":
            #Segment must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "borrower_data_issue_id","Segment",
                                                                                "MissingValue",processSummaryId)

        #As regions are combined in the sample data e.g Africa/Asia it seems overcomplex to validate them other than as non-blank
        #Other nomenclature may be needed in future such as BRICS, APAC etc and then this would need to be adjusted
        #if hard continents/continent pairs were used
        
        if row.region.strip()=="":
            #Segment must be non-blank
            (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                "borrower_data_issue_id","Region","MissingValue",processSummaryId)

        percentageTotal=0
        percentErrorRaised=False
        
        #Validate country/revenue country pairs
        for i in range(1,borrowerMaxCountries):
            #The country names will start at index position 6
            #The revenue percentages for each country will start at 6 + borrowerMaxCountries 

            #Again, oddly sometimes blanks in Excel are python floats....
            if isinstance(row[5 + i],(int,float)):
                countryName=""
            else:
                countryName=row[5 + i].strip()
                
            revenuePercent=row[5 + borrowerMaxCountries + i]

            if(countryName!="" and revenuePercent==""):
                #Country with no revenue %
                (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                    "borrower_data_issue_id", f"Revenue Country {i}",
                                                                                    "MissingValue",processSummaryId)
            elif (countryName=="" and revenuePercent!=""):
                #Revenue % but no country
                (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                    "borrower_data_issue_id",f"Country {i}",
                                                                                    "MissingValue",processSummaryId)
            elif(countryName!="" and revenuePercent!=""):
                #They both have value.
                #Is the country in the ISO list?

                if countryName.lower() not in valid_countries:
                    (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                        "borrower_data_issue_id",f"Country {i}",
                                                                                        "ValueOutOfRange",processSummaryId)

                if validatePercentage(revenuePercent)!="":
                    (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                        "borrower_data_issue_id",f"Revenue Country {i}",
                                                                                        validatePercentage(revenuePercent),processSummaryId)
                else:
                    #Running total of revenue percentage
                    percentageTotal+=revenuePercent

                    #Log issue if percent out of range, but only log it once per row
                    if(percentageTotal<0  or percentageTotal>100) and not percentErrorRaised:
                       (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,issueRowInserted,row,issueId,
                                                                                           "borrower_data_issue_id",f"Revenue Country {i}",
                                                                                           "ValueOutOfRange",processSummaryId)
                        
        if issueRowInserted:
            issueId+=1

    #Now that we have processed all the rows, look for duplicates on borrower name

    for duplicate in borrower_df[borrower_df.duplicated(subset=['borrower'],keep='last')].itertuples():

        #Insert into issue tables as before, but detail table will have a single row with a blank fieldname
        (issueRowInserted,issues_df,issuedesc_df)=insertIssueIntoDataframes(issues_df,issuedesc_df,False,duplicate,issueId,
                                                                            "borrower_data_issue_id","","Duplicate",processSummaryId)

        if issueRowInserted:
            issueId+=1

    #We now have our errors saved into our 2 dataframes - 
    #Save them into our SQL tables
    
    if issues_df.shape[0]>0:

        engine=create_engine(connect_string)
        
        #Save issues and detail/description tables to db
        issues_df.to_sql(borrowerIssueTable, engine, index=False,if_exists='append',schema='data_quality')
        issuedesc_df.to_sql(borrowerIssueDetailTable,engine, index=False,if_exists='append',schema='data_quality')
        
    return issueId-1


Various utility functions to manage storage of the validation issues are defined below

In [69]:
#Some utility functions to enable code re-use

#This function is to avoid repeating the same lines ad nauseam ...
#issues_df = the dataframe where each row with an error is stored
#issue_desc_df = the dataframe where a row with the description of the issue for each field with an issue is stored
#issueRowInserted - do we need to insert a new issue row? - will be true if we havent already logged an issue for this row
#row - a namedtuple for the row which can be used like value=row.somefield
#issueId the current issue id we are working with/creating
#issuedesc_foreignkey - the fieldname of the foreign key to the issue table in the description table
#fieldName - the name of the field with the error e.g "Borrower"
#A description of the error e.g "Missing value"
#Returns a bool - have we inserted an issue row for this row in the df?

def insertIssueIntoDataframes(issues_df:pd.DataFrame,issue_desc_df:pd.DataFrame,issueRowInserted:bool,row:namedtuple,issueId:int,
                             issuedesc_foreignkey:str,fieldName:str,issueDesc:str,processSummaryId:int)->bool:

    if not issueRowInserted:

            #We haven't saved the issue row yet, so save it

            #Create a temp dataframe to append with this row only and set the column names
            #row will not process summary id so leave it out initially to enable us to just use the row to set everything
            temp_df=pd.DataFrame([row],columns=issues_df.columns[~issues_df.columns.isin(['process_summary_id'])])
            

            #Set id and process summary id fields so the right relationship is there
            temp_df["id"]=issueId
            temp_df["process_summary_id"]=processSummaryId

            issues_df=pd.concat([issues_df,temp_df],ignore_index=True)

            issueRowInserted=True

    #Add a new description row for the error for this field, with a foreign key tp the issue id
    newRow={issuedesc_foreignkey:issueId,"field_name":fieldName,"description":issueDesc};
    temp_df=pd.DataFrame([newRow])
    issue_desc_df=pd.concat([issue_desc_df,temp_df],ignore_index=True)

    return (issueRowInserted,issues_df,issue_desc_df)

#Takes a report date as a  string and validates that it is in dd/mm/yyyy format
#and that the component parts of the date are within the expected bounds
#Returns a message which can be inserted into the error description field directly

def validateReportDate(inputDate)->str:

    test:datetime.date

    if not isinstance(inputDate,datetime):
        
        #if input is not a date, check format: if a number reject, if a string try and parse

        if isinstance(inputDate,(int,float)):
            return "InvalidDateFormat"
    
        #Is the date format valid?
        try:
            test=datetime.strptime(inputDate,"%d/%m/%Y")
        except ValueError:
            return "InvalidDateFormat"
        
    #Check that the report date falls between 01/01/1970 and 01/01/2050 

    if inputDate<datetime.strptime("01/01/2000", r"%d/%m/%Y") or inputDate>datetime.strptime("01/01/2050", r"%d/%m/%Y"):
        return "DateOutOfRange"

    return ""

def validatePercentage(inputValue:str)->str:
    regEx=r"\d{2}.\d{2}%" #Regular expression - 2 numbers, a period, 2 numbers then a percent sign

    #If a number is passed...
    if isinstance(inputValue,(int,float)):
        if inputValue<0 or inputValue>100:
            return "InvalidPercentageFormat"
    else:
        #If it is a string use the RegEx
        if not re.match(regEx, inputValue):
            return "InvalidPercentageFormat"

    return ""

In [70]:
#Log an entry in process summaries with the current date/time and the error stats
#This table is not cleared down so that we keep a history
def logProcessSummary(validRows:int,rowsWithIssues:int)->int:
    
    now=datetime.now()
    
    engine=create_engine(connect_string)
    
    #Get the process_summaries table metadata so it can be used to insert       
    #Use quoted name to stop it doing "data_quality.table_name" not "data_quality"."table_name"
    summaryTable=Table(f"process_summaries", metadata_obj, autoload_with=engine,
                       schema=quoted_name(dataQualitySchema, quote=False))

    schema=quoted_name("dev.exchange_rates", quote=False)

    #Use sqlAlchemy to build the parameterised insert statement 
    stmt = insert(summaryTable).values(run_date_time=now, valid_rows=validRows,rows_with_issues=rowsWithIssues)

    with engine.connect() as conn:
      result = conn.execute(stmt)
      conn.commit()
      return result.inserted_primary_key[0] # Return id of new row
    

In [71]:
def updateProcessSummary(id:int,validRows:int,rowsWithIssues:int):

    now=datetime.now()
    
    engine=create_engine(connect_string)
    
    #Get the process_summaries table metadata so it can be used to update                               
    summaryTable=Table(f"process_summaries", metadata_obj, autoload_with=engine,
               schema=quoted_name(dataQualitySchema, quote=False))        

    stmt = (
     update(summaryTable)
     .where(summaryTable.c.id == id)
     .values(run_date_time=now,valid_rows=validRows,rows_with_issues=rowsWithIssues)
    )

    with engine.connect() as conn:
      result = conn.execute(stmt)
      conn.commit()

The code below represents the highest level of the process, calling the functions above as needed.

In [73]:
#Load data
portfolio_df=getPortfolioData()
coupon_df=getCouponData()
borrower_df=getBorrowerData()
valuation_df=getValuationData()

rowsWithIssues=0

#Define sqlalchemy metadata here so we can use it everywhere
metadata_obj = MetaData()

#Get a count of all rows in the source (both with and without issues)
stagingRowCount=portfolio_df.shape[0] + coupon_df.shape[0] + borrower_df.shape[0] + valuation_df.shape[0]

#Get the list of valid ISO country codes for validation later
country_df=getISOCountryData()

#Clear down existing data
clearDownData()

#Create a new process summary with zero rowcounts, storing the id for later..
processSummaryId=logProcessSummary(0,0)

#Perform data validation, generating the raw data for the data quality report into the data_issues and data_issue_details tables
rowsWithIssues+=performValidation(portfolio_df,portfolioSourceName,processSummaryId)
rowsWithIssues+=performValidation(coupon_df,couponSourceName,processSummaryId)

#Borrower validation needs countries
rowsWithIssues+=performValidation(borrower_df,borrowerSourceName,processSummaryId,country_df)
rowsWithIssues+=performValidation(valuation_df,valuationSourceName,processSummaryId)

#Remove/fix issues if we agree to do that to clean data??

#Update summary with counts
updateProcessSummary(processSummaryId,stagingRowCount-rowsWithIssues,rowsWithIssues)

engine=create_engine(connect_string)
#Load data into staging tables

portfolio_df.to_sql("portfolios", engine, index=False,if_exists='append',schema='staging')
coupon_df.to_sql("coupons", engine, index=False,if_exists='append',schema='staging')
borrower_df.to_sql("borrowers", engine, index=False,if_exists='append',schema='staging')
valuation_df.to_sql("valuations", engine, index=False,if_exists='append',schema='staging')

#Call stored procedure to transform data into relational structure in the warehouse.

with engine.connect() as conn:
    conn.execute(text("CALL warehouse.sp_reload_warehouse()"))
